# 02 – Data Cleaning

Handle missing values, duplicates, invalid types, outliers, and compute derived columns.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from ml.preprocessing import load_and_clean, compute_engagement_rate, extract_posting_hour

raw = pd.read_csv('../data/sample/sample_posts.csv')
print(f'Raw rows: {len(raw)}')
raw.head(3)

In [ ]:
# Step 1: Remove duplicates
deduped = raw.drop_duplicates()
print(f'After dedup: {len(deduped)} rows (removed {len(raw) - len(deduped)})')

In [ ]:
# Step 2: Coerce numerics and fill engagement nulls
numeric_cols = ['likes','comments','shares','saves','reach','impressions','views',
                'caption_length','hashtag_count','followers']
for col in numeric_cols:
    deduped[col] = pd.to_numeric(deduped[col], errors='coerce')

for col in ['likes','comments','shares','saves']:
    deduped[col] = deduped[col].fillna(0)

print('Missing after fill:')
print(deduped[numeric_cols].isnull().sum())

In [ ]:
# Step 3: Drop rows with zero reach (can't compute engagement rate)
valid = deduped[deduped['reach'].notna() & (deduped['reach'] > 0)].copy()
print(f'After reach filter: {len(valid)} rows')

In [ ]:
# Step 4: Compute engagement_rate and posting_hour
valid['engagement_rate'] = compute_engagement_rate(valid)
valid['posting_hour'] = extract_posting_hour(valid['posting_time'])

print('Engagement rate stats:')
valid['engagement_rate'].describe()

In [ ]:
# Step 5: Use the full pipeline
df_clean = load_and_clean('../data/sample/sample_posts.csv')
print(f'Final clean dataset: {len(df_clean)} rows')
print(df_clean['performance_label'].value_counts())